In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

# 경로 설정
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'

class OpsinosisAnalyzer:
    """
    Opinosis Opinion Dataset 전용 분석기
    - 제품별 리뷰 클러스터링
    - Affinity Propagation 기반 유사도 분석
    """
    
    def __init__(self, data_path):
        """
        Parameters:
        -----------
        data_path : str
            Opinosis 데이터셋 경로 (topics 폴더 포함)
        """
        self.data_path = Path(data_path)
        self.documents = []
        self.metadata = []  # 제품명, 카테고리 등 메타데이터
        self.reviews_by_product = defaultdict(list)
        
    def load_opinosis_data(self):
        """
        Opinosis 데이터셋 로드
        
        데이터 구조:
        - data/
          - product1_name.txt.data
          - product2_name.txt.data
          - ...
        
        각 파일: 한 줄에 하나의 리뷰 요약
        """
        print("="*80)
        print("📂 Opinosis 데이터셋 로딩 중...")
        print("="*80)
        
        # topics 폴더 찾기
        data_path = self.data_path 
        all_files = glob.glob(os.path.join(self.data_path, "*.data"))
        
        if not topics_path.exists():
            # 현재 경로가 topics일 수도 있음
            if self.data_path.name == 'data':
                topics_path = self.data_path
            else:
                raise FileNotFoundError(f"topics 폴더를 찾을 수 없습니다: {topics_path}")
        
        # 모든 .doc 파일 읽기
        txt_files = list(topics_path.glob('*.data'))
        
        if not txt_files:
            raise FileNotFoundError(f"topics 폴더에 .data 파일이 없습니다: {topics_path}")
        
        print(f"✅ 발견된 제품 파일: {len(txt_files)}개\n")
        
        for file_path in sorted(txt_files):
            product_name = file_path.stem  # 파일명 (확장자 제외)
            
            # 제품 카테고리 추출 (파일명 패턴 기반)
            category = self._extract_category(product_name)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    reviews = f.readlines()
                
                # 빈 줄 제거 및 정리
                reviews = [r.strip() for r in reviews if r.strip()]
                
                print(f"📄 {product_name}")
                print(f"   - 카테고리: {category}")
                print(f"   - 리뷰 수: {len(reviews)}")
                
                # 문서 및 메타데이터 저장
                for review in reviews:
                    self.documents.append(review)
                    self.metadata.append({
                        'product': product_name,
                        'category': category,
                        'file': file_path.name
                    })
                    self.reviews_by_product[product_name].append(review)
                
            except Exception as e:
                print(f"❌ 파일 읽기 오류 ({file_path.name}): {e}")
        
        print(f"\n{'='*80}")
        print(f"✅ 데이터 로딩 완료")
        print(f"   - 총 제품 수: {len(self.reviews_by_product)}")
        print(f"   - 총 리뷰 수: {len(self.documents)}")
        print(f"{'='*80}\n")
        
        return self.documents, self.metadata
    
    def _extract_category(self, product_name):
        """
        제품명에서 카테고리 추출
        
        Opinosis 데이터셋의 일반적인 패턴:
        - hotel_name -> hotel
        - car_name -> car
        - electronics_name -> electronics
        """
        # 언더스코어로 분리하여 첫 단어 추출
        parts = product_name.split('_')
        
        # 일반적인 카테고리 키워드
        categories = ['hotel', 'car', 'phone', 'camera', 'laptop', 
                     'mp3', 'tv', 'tablet', 'router', 'speaker']
        
        for part in parts:
            part_lower = part.lower()
            for cat in categories:
                if cat in part_lower:
                    return cat
        
        # 매칭 안되면 첫 단어 반환
        return parts[0] if parts else 'unknown'
    
    def get_data_summary(self):
        """
        데이터 요약 통계
        """
        df = pd.DataFrame(self.metadata)
        df['review_length'] = [len(doc.split()) for doc in self.documents]
        
        print("\n" + "="*80)
        print("📊 데이터 요약 통계")
        print("="*80)
        
        # 제품별 통계
        product_stats = df.groupby('product').agg({
            'review_length': ['count', 'mean', 'min', 'max']
        }).round(2)
        
        print("\n[제품별 리뷰 통계]")
        print(product_stats.to_string())
        
        # 카테고리별 통계
        category_stats = df.groupby('category').agg({
            'review_length': ['count', 'mean']
        }).round(2)
        
        print("\n[카테고리별 통계]")
        print(category_stats.to_string())
        
        # 전체 통계
        print("\n[전체 통계]")
        print(f"  - 총 리뷰 수: {len(self.documents)}")
        print(f"  - 평균 리뷰 길이: {df['review_length'].mean():.2f} 단어")
        print(f"  - 최소 리뷰 길이: {df['review_length'].min()} 단어")
        print(f"  - 최대 리뷰 길이: {df['review_length'].max()} 단어")
        print("="*80 + "\n")
        
        return df


class OpsinosisAPAnalyzer(OpsinosisAnalyzer):
    """
    Opinosis + Affinity Propagation 통합 분석기
    """
    
    def __init__(self, data_path, damping=0.9, max_iter=500, preference=None):
        super().__init__(data_path)
        self.damping = damping
        self.max_iter = max_iter
        self.preference = preference
        self.vectorizer = None
        self.tfidf_matrix = None
        self.ap_model = None
        self.labels = None
        self.exemplars = None
    
    def preprocess_and_vectorize(self, max_features=500, ngram_range=(1, 2),
                                 min_df=2, max_df=0.8):
        """
        TF-IDF 벡터화
        """
        print("="*80)
        print("🔧 TF-IDF 벡터화 중...")
        print("="*80)
        
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            stop_words='english',
            lowercase=True,
            strip_accents='unicode'
        )
        
        self.tfidf_matrix = self.vectorizer.fit_transform(self.documents)
        
        print(f"✅ 벡터화 완료")
        print(f"   - 문서 수: {self.tfidf_matrix.shape[0]}")
        print(f"   - 피처 수: {self.tfidf_matrix.shape[1]}")
        print(f"   - Sparsity: {(1.0 - self.tfidf_matrix.nnz / (self.tfidf_matrix.shape[0] * self.tfidf_matrix.shape[1])) * 100:.2f}%")
        print("="*80 + "\n")
        
        return self.tfidf_matrix
    
    def fit_affinity_propagation(self, similarity_metric='cosine'):
        """
        Affinity Propagation 클러스터링
        """
        print("="*80)
        print("🎯 Affinity Propagation 클러스터링 중...")
        print("="*80)
        
        # 유사도 행렬 계산
        if similarity_metric == 'cosine':
            similarity_matrix = cosine_similarity(self.tfidf_matrix)
        else:
            from sklearn.metrics.pairwise import euclidean_distances
            distance_matrix = euclidean_distances(self.tfidf_matrix)
            similarity_matrix = -distance_matrix
        
        # Preference 설정
        if self.preference is None:
            self.preference = np.median(similarity_matrix)
        
        print(f"   - Damping: {self.damping}")
        print(f"   - Max Iterations: {self.max_iter}")
        print(f"   - Preference: {self.preference:.4f}")
        print(f"   - Similarity Metric: {similarity_metric}\n")
        
        # AP 모델 학습
        self.ap_model = AffinityPropagation(
            damping=self.damping,
            max_iter=self.max_iter,
            preference=self.preference,
            affinity='precomputed',
            random_state=42,
            verbose=False
        )
        
        self.labels = self.ap_model.fit_predict(similarity_matrix)
        self.exemplars = self.ap_model.cluster_centers_indices_
        
        n_clusters = len(self.exemplars)
        
        print(f"✅ 클러스터링 완료")
        print(f"   - 발견된 군집 수: {n_clusters}")
        print(f"   - Exemplar 수: {len(self.exemplars)}")
        print(f"   - 반복 횟수: {self.ap_model.n_iter_}")
        print("="*80 + "\n")
        
        # 군집 분포
        self._print_cluster_distribution()
        
        return self.labels
    
    def _print_cluster_distribution(self):
        """
        군집 분포 출력
        """
        from collections import Counter
        
        cluster_counts = Counter(self.labels)
        
        print("📊 군집 분포:")
        for cluster_id in sorted(cluster_counts.keys()):
            count = cluster_counts[cluster_id]
            percentage = (count / len(self.labels)) * 100
            print(f"   - 군집 {cluster_id}: {count}개 ({percentage:.1f}%)")
        print()
    
    def analyze_clusters_by_product(self):
        """
        제품별 군집 분석
        """
        print("="*80)
        print("🔍 제품별 군집 분석")
        print("="*80 + "\n")
        
        df = pd.DataFrame({
            'product': [m['product'] for m in self.metadata],
            'category': [m['category'] for m in self.metadata],
            'cluster': self.labels,
            'review': self.documents
        })
        
        # 제품별 군집 분포
        product_cluster_dist = df.groupby(['product', 'cluster']).size().unstack(fill_value=0)
        
        print("[제품별 군집 분포]")
        print(product_cluster_dist.to_string())
        print()
        
        # 카테고리별 군집 분포
        category_cluster_dist = df.groupby(['category', 'cluster']).size().unstack(fill_value=0)
        
        print("[카테고리별 군집 분포]")
        print(category_cluster_dist.to_string())
        print()
        
        return df
    
    def get_cluster_keywords(self, cluster_id, top_n=10):
        """
        특정 군집의 키워드 추출
        """
        # 해당 군집의 문서 인덱스
        cluster_docs = np.where(self.labels == cluster_id)[0]
        
        # 군집 내 문서들의 평균 TF-IDF
        cluster_tfidf = self.tfidf_matrix[cluster_docs].mean(axis=0).A1
        
        # 상위 키워드
        feature_names = self.vectorizer.get_feature_names_out()
        top_indices = cluster_tfidf.argsort()[-top_n:][::-1]
        top_keywords = [(feature_names[i], cluster_tfidf[i]) for i in top_indices]
        
        return top_keywords
    
    def get_cluster_summary(self, cluster_id):
        """
        군집 요약 정보
        """
        cluster_docs = np.where(self.labels == cluster_id)[0]
        exemplar_idx = self.exemplars[cluster_id]
        
        # 해당 군집의 제품 분포
        cluster_products = [self.metadata[i]['product'] for i in cluster_docs]
        product_counts = pd.Series(cluster_products).value_counts()
        
        summary = {
            'cluster_id': cluster_id,
            'n_documents': len(cluster_docs),
            'exemplar_idx': exemplar_idx,
            'exemplar_review': self.documents[exemplar_idx],
            'exemplar_product': self.metadata[exemplar_idx]['product'],
            'top_keywords': self.get_cluster_keywords(cluster_id, 10),
            'product_distribution': product_counts.to_dict(),
            'sample_reviews': [self.documents[i] for i in cluster_docs[:5]]
        }
        
        return summary
    
    def print_cluster_report(self):
        """
        전체 군집 리포트 출력
        """
        print("\n" + "="*80)
        print("📋 전체 군집 리포트")
        print("="*80 + "\n")
        
        n_clusters = len(self.exemplars)
        
        for cluster_id in range(n_clusters):
            summary = self.get_cluster_summary(cluster_id)
            
            print(f"[군집 {cluster_id}]")
            print(f"  문서 수: {summary['n_documents']}")
            print(f"  Exemplar 제품: {summary['exemplar_product']}")
            print(f"  Exemplar 리뷰: {summary['exemplar_review'][:100]}...")
            print(f"\n  상위 키워드:")
            for word, score in summary['top_keywords'][:5]:
                print(f"    - {word}: {score:.4f}")
            print(f"\n  제품 분포:")
            for product, count in list(summary['product_distribution'].items())[:3]:
                print(f"    - {product}: {count}개")
            print("\n" + "-"*80 + "\n")
    
    def visualize_clusters_2d(self, method='tsne', color_by='cluster',
                               figsize=(14, 10), save_path=None):
        """
        2D 군집 시각화
        
        Parameters:
        -----------
        method : str
            'tsne' or 'pca'
        color_by : str
            'cluster', 'product', 'category'
        """
        print(f"🎨 {method.upper()} 차원 축소 중...")
        
        # 차원 축소
        if method == 'tsne':
            reducer = TSNE(n_components=2, random_state=42, 
                          perplexity=min(30, len(self.documents)-1))
        else:
            reducer = PCA(n_components=2, random_state=42)
        
        coords = reducer.fit_transform(self.tfidf_matrix.toarray())
        
        # 시각화
        fig, ax = plt.subplots(figsize=figsize)
        
        if color_by == 'cluster':
            # 군집별 색상
            n_clusters = len(set(self.labels))
            colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
            
            for cluster_id in range(n_clusters):
                mask = self.labels == cluster_id
                cluster_coords = coords[mask]
                
                ax.scatter(cluster_coords[:, 0], cluster_coords[:, 1],
                          c=[colors[cluster_id]], label=f'군집 {cluster_id}',
                          alpha=0.6, s=80)
                
                # Exemplar 표시
                exemplar_idx = self.exemplars[cluster_id]
                exemplar_coord = coords[exemplar_idx]
                ax.scatter(exemplar_coord[0], exemplar_coord[1],
                          c=[colors[cluster_id]], marker='*', s=500,
                          edgecolors='black', linewidths=2, zorder=10)
            
            ax.set_title(f'AP 클러스터링 결과 ({method.upper()})', 
                        fontsize=16, fontweight='bold')
        
        elif color_by == 'product':
            # 제품별 색상
            products = [m['product'] for m in self.metadata]
            unique_products = sorted(set(products))
            n_products = len(unique_products)
            colors = plt.cm.tab20(np.linspace(0, 1, n_products))
            color_map = dict(zip(unique_products, colors))
            
            for product in unique_products:
                mask = [m['product'] == product for m in self.metadata]
                product_coords = coords[mask]
                
                ax.scatter(product_coords[:, 0], product_coords[:, 1],
                          c=[color_map[product]], label=product,
                          alpha=0.6, s=80)
            
            ax.set_title(f'제품별 리뷰 분포 ({method.upper()})', 
                        fontsize=16, fontweight='bold')
        
        elif color_by == 'category':
            # 카테고리별 색상
            categories = [m['category'] for m in self.metadata]
            unique_categories = sorted(set(categories))
            n_categories = len(unique_categories)
            colors = plt.cm.Set3(np.linspace(0, 1, n_categories))
            color_map = dict(zip(unique_categories, colors))
            
            for category in unique_categories:
                mask = [m['category'] == category for m in self.metadata]
                category_coords = coords[mask]
                
                ax.scatter(category_coords[:, 0], category_coords[:, 1],
                          c=[color_map[category]], label=category,
                          alpha=0.6, s=80)
            
            ax.set_title(f'카테고리별 리뷰 분포 ({method.upper()})', 
                        fontsize=16, fontweight='bold')
        
        ax.set_xlabel(f'{method.upper()} 1', fontsize=12)
        ax.set_ylabel(f'{method.upper()} 2', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"💾 그래프 저장: {save_path}")
        
        plt.show()
    
    def visualize_product_cluster_heatmap(self, figsize=(12, 8), save_path=None):
        """
        제품 vs 군집 히트맵
        """
        df = pd.DataFrame({
            'product': [m['product'] for m in self.metadata],
            'cluster': self.labels
        })
        
        # 크로스탭
        crosstab = pd.crosstab(df['product'], df['cluster'], normalize='index') * 100
        
        fig, ax = plt.subplots(figsize=figsize)
        
        sns.heatmap(crosstab, annot=True, fmt='.1f', cmap='YlOrRd',
                   cbar_kws={'label': '비율 (%)'}, ax=ax)
        
        ax.set_title('제품별 군집 분포 (%)', fontsize=16, fontweight='bold')
        ax.set_xlabel('군집 ID', fontsize=12)
        ax.set_ylabel('제품', fontsize=12)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"💾 그래프 저장: {save_path}")
        
        plt.show()
    
    def find_similar_reviews(self, review_idx, top_n=5, same_product_only=False):
        """
        유사 리뷰 찾기
        
        Parameters:
        -----------
        review_idx : int
            기준 리뷰 인덱스
        top_n : int
            상위 N개
        same_product_only : bool
            같은 제품 내에서만 검색
        """
        review_vec = self.tfidf_matrix[review_idx]
        similarities = cosine_similarity(review_vec, self.tfidf_matrix)[0]
        
        # 자기 자신 제외
        similarities[review_idx] = -1
        
        if same_product_only:
            # 같은 제품만
            product = self.metadata[review_idx]['product']
            mask = np.array([m['product'] == product for m in self.metadata])
            similarities[~mask] = -1
        
        # 상위 N개
        top_indices = similarities.argsort()[-top_n:][::-1]
        
        similar_reviews = []
        for idx in top_indices:
            similar_reviews.append({
                'index': idx,
                'similarity': similarities[idx],
                'product': self.metadata[idx]['product'],
                'category': self.metadata[idx]['category'],
                'cluster': self.labels[idx],
                'review': self.documents[idx]
            })
        
        return similar_reviews
    
    def export_results(self, output_dir='opinosis_ap_results'):
        """
        결과 내보내기
        """
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        print(f"\n📦 결과 내보내기 중... ({output_dir})")
        
        # 1. 군집 결과 CSV
        df = pd.DataFrame({
            'review': self.documents,
            'product': [m['product'] for m in self.metadata],
            'category': [m['category'] for m in self.metadata],
            'cluster': self.labels
        })
        df.to_csv(output_path / 'clustering_results.csv', index=False, encoding='utf-8-sig')
        print(f"   ✅ clustering_results.csv")
        
        # 2. 군집 요약 리포트
        with open(output_path / 'cluster_summary.txt', 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("Opinosis AP 클러스터링 요약 리포트\n")
            f.write("="*80 + "\n\n")
            
            for cluster_id in range(len(self.exemplars)):
                summary = self.get_cluster_summary(cluster_id)
                
                f.write(f"[군집 {cluster_id}]\n")
                f.write(f"  문서 수: {summary['n_documents']}\n")
                f.write(f"  Exemplar 제품: {summary['exemplar_product']}\n")
                f.write(f"  Exemplar 리뷰: {summary['exemplar_review']}\n\n")
                f.write(f"  상위 키워드:\n")
                for word, score in summary['top_keywords']:
                    f.write(f"    - {word}: {score:.4f}\n")
                f.write(f"\n  제품 분포:\n")
                for product, count in summary['product_distribution'].items():
                    f.write(f"    - {product}: {count}개\n")
                f.write("\n" + "-"*80 + "\n\n")
        
        print(f"   ✅ cluster_summary.txt")
        
        # 3. 제품별 군집 분포
        product_cluster = df.groupby(['product', 'cluster']).size().unstack(fill_value=0)
        product_cluster.to_csv(output_path / 'product_cluster_distribution.csv', encoding='utf-8-sig')
        print(f"   ✅ product_cluster_distribution.csv")
        
        print(f"\n✅ 모든 결과가 '{output_dir}' 폴더에 저장되었습니다.\n")


# ==================== 메인 실행 코드 ====================

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("🎯 Opinosis 데이터셋 + Affinity Propagation 분석")
    print("="*80 + "\n")
    
    # 1. 데이터 경로 설정 (실제 경로로 변경 필요)
    DATA_PATH = "../data"  # 또는 실제 경로
    
    # 2. 분석기 초기화
    analyzer = OpsinosisAPAnalyzer(
        data_path=DATA_PATH,
        damping=0.9,
        max_iter=500,
        preference=None  # 자동 설정
    )
    
    # 3. 데이터 로드
    documents, metadata = analyzer.load_opinosis_data()
    
    # 4. 데이터 요약
    df_summary = analyzer.get_data_summary()
    
    # 5. TF-IDF 벡터화
    tfidf_matrix = analyzer.preprocess_and_vectorize(
        max_features=500,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.8
    )
    
    # 6. AP 클러스터링
    labels = analyzer.fit_affinity_propagation(similarity_metric='cosine')
    
    # 7. 제품별 군집 분석
    df_clusters = analyzer.analyze_clusters_by_product()
    
    # 8. 군집 리포트 출력
    analyzer.print_cluster_report()
    
    # 9. 시각화
    print("\n" + "="*80)
    print("📊 시각화 생성 중...")
    print("="*80 + "\n")
    
    # 9-1. 군집별 시각화 (t-SNE)
    analyzer.visualize_clusters_2d(
        method='tsne',
        color_by='cluster',
        save_path='opinosis_clusters_tsne.png'
    )
    
    # 9-2. 제품별 시각화
    analyzer.visualize_clusters_2d(
        method='tsne',
        color_by='product',
        save_path='opinosis_products_tsne.png'
    )
    
    # 9-3. 제품-군집 히트맵
    analyzer.visualize_product_cluster_heatmap(
        save_path='opinosis_product_cluster_heatmap.png'
    )
    
    # 10. 유사 리뷰 찾기 예시
    print("\n" + "="*80)
    print("🔍 유사 리뷰 검색 예시")
    print("="*80 + "\n")
    
    sample_idx = 0
    print(f"기준 리뷰 (인덱스 {sample_idx}):")
    print(f"  제품: {metadata[sample_idx]['product']}")
    print(f"  내용: {documents[sample_idx]}\n")
    
    similar_reviews = analyzer.find_similar_reviews(sample_idx, top_n=5)
    
    print("유사한 리뷰:")
    for i, sim_review in enumerate(similar_reviews, 1):
        print(f"\n{i}. 유사도: {sim_review['similarity']:.4f}")
        print(f"   제품: {sim_review['product']}")
        print(f"   군집: {sim_review['cluster']}")
        print(f"   내용: {sim_review['review'][:100]}...")
    
    # 11. 결과 내보내기
    analyzer.export_results(output_dir='opinosis_ap_results')
    
    print("\n" + "="*80)
    print("✅ 모든 분석 완료!")
    print("="*80 + "\n")



In [ ]:
from utils.preprocessing import load_file_data


# 1. 데이터 로드 (Opinosis 데이터셋 예시)
# 실제로는 파일에서 읽어와야 함
documents_df = load_file_data()
print(f"✅ {len(documents_df)}개 문서 로드 완료\n")

#